In [4]:
# ============================================================
# RAG STEP 2: STORING CHUNKS IN CHROMADB
# Complete Google Colab - Single Cell
# ============================================================

# ------------------------------------------------------------
# 1. INSTALL REQUIRED LIBRARIES
# ------------------------------------------------------------
!pip install -q langchain-text-splitters chromadb sentence-transformers

# ------------------------------------------------------------
# 2. IMPORT LIBRARIES
# ------------------------------------------------------------
import os
import chromadb
from chromadb.utils import embedding_functions
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ------------------------------------------------------------
# 3. CREATE SAMPLE DOCUMENT
# ------------------------------------------------------------
os.makedirs("docs", exist_ok=True)

article_text = """Artificial Intelligence: An Overview

Artificial Intelligence, or AI, is the field of computer science focused on building systems
that can perform tasks which normally require human intelligence. These tasks include
understanding language, recognizing images, making decisions, and learning from experience.

Machine Learning is a subset of AI where systems learn patterns from data instead of following
hardcoded rules. Instead of programming every decision by hand, we feed the system many examples,
and it learns the underlying pattern on its own. This approach powers everything from spam filters
to recommendation systems.

Deep Learning takes Machine Learning further by using neural networks with many layers. These
layered networks can automatically discover complex patterns in large amounts of data, such as
recognizing faces in photos or understanding the meaning of a sentence.

Natural Language Processing, or NLP, is the branch of AI that focuses specifically on human
language. NLP powers chatbots, translation tools, and search engines. A key building block of
modern NLP is the embedding, which converts words or sentences into numeric vectors that capture
meaning.

Retrieval-Augmented Generation, or RAG, is a technique that combines a search system with a
language model. Instead of relying only on what a language model memorized during training, RAG
first retrieves relevant, up-to-date information from a document collection, then passes that
information to the model so it can generate a more accurate, grounded answer.

Vector Databases store embeddings and allow fast similarity search across millions of documents.
They are a core infrastructure piece behind modern RAG systems, chatbots, and recommendation
engines used by companies around the world today.
"""

with open("docs/ai_overview.txt", "w") as f:
    f.write(article_text)

print("Document created: docs/ai_overview.txt")

# ------------------------------------------------------------
# 4. CHUNK THE DOCUMENT
# ------------------------------------------------------------
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = splitter.split_text(article_text)

print("\nNumber of chunks created:", len(chunks))
for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i} ({len(chunk)} chars) ---")
    print(chunk)

# ------------------------------------------------------------
# 5. CREATE METADATA AND IDS
# ------------------------------------------------------------
chunk_metadatas = [
    {
        "source": "ai_overview.txt",
        "chunk_index": idx,
        "char_count": len(c)
    }
    for idx, c in enumerate(chunks)
]

chunk_ids = [
    f"ai_overview_chunk_{idx}"
    for idx in range(len(chunks))
]

print("\nExample metadata for chunk 0:")
print(chunk_metadatas[0])

print("\nExample ID for chunk 0:")
print(chunk_ids[0])

# ------------------------------------------------------------
# 6. SET UP CHROMADB
# ------------------------------------------------------------
client = chromadb.Client()

hf_embedder = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

collection = client.create_collection(
    name="rag_chunks",
    embedding_function=hf_embedder
)

print("\nCollection created:", collection.name)

# ------------------------------------------------------------
# 7. STORE CHUNKS IN CHROMADB
# ------------------------------------------------------------
collection.add(
    documents=chunks,
    metadatas=chunk_metadatas,
    ids=chunk_ids
)

print("\nTotal chunks stored in ChromaDB:", collection.count())

# ------------------------------------------------------------
# 8. VERIFY STORED CHUNKS
# ------------------------------------------------------------
stored = collection.get(
    ids=chunk_ids,
    include=["documents", "metadatas"]
)

print("\n========== STORED CHUNKS ==========")
for cid, doc, meta in zip(stored["ids"], stored["documents"], stored["metadatas"]):
    print(f"ID: {cid}")
    print(f"Metadata: {meta}")
    print(f"Text: {doc[:100]}...")
    print("-" * 60)

# ------------------------------------------------------------
# 9. QUERY STORED CHUNKS
# ------------------------------------------------------------
print("\n========== RAG QUERY 1 ==========")
results = collection.query(
    query_texts=["How does RAG combine search with language models?"],
    n_results=2,
    include=["documents", "metadatas", "distances"]
)

for doc, meta, distance in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
    print(f"Distance: {distance:.4f} | Source: {meta['source']} | Chunk #{meta['chunk_index']}")
    print(doc)
    print("-" * 60)

# ------------------------------------------------------------
# 10. CREATE SECOND DOCUMENT
# ------------------------------------------------------------
second_doc_text = """Embeddings and Vector Search in Practice

Modern search engines increasingly rely on embeddings rather than simple keyword matching.
An embedding model converts a search query into the same vector space as the stored documents,
allowing the system to find results that match the meaning of the query, not just its exact words.

This is especially powerful for customer support systems, where a user might phrase a question very
differently from how it appears in a help article, yet still expect to find the right answer.
"""

with open("docs/embeddings_in_practice.txt", "w") as f:
    f.write(second_doc_text)

# ------------------------------------------------------------
# 11. CHUNK SECOND DOCUMENT
# ------------------------------------------------------------
second_doc_chunks = splitter.split_text(second_doc_text)

second_doc_metadatas = [
    {
        "source": "embeddings_in_practice.txt",
        "chunk_index": idx,
        "char_count": len(c)
    }
    for idx, c in enumerate(second_doc_chunks)
]

second_doc_ids = [
    f"embeddings_in_practice_chunk_{idx}"
    for idx in range(len(second_doc_chunks))
]

# ------------------------------------------------------------
# 12. ADD SECOND DOCUMENT TO SAME COLLECTION
# ------------------------------------------------------------
collection.add(
    documents=second_doc_chunks,
    metadatas=second_doc_metadatas,
    ids=second_doc_ids
)

print("\nChunks from second document added:", len(second_doc_chunks))
print("Total chunks now stored in ChromaDB:", collection.count())

# ------------------------------------------------------------
# 13. QUERY ACROSS BOTH DOCUMENTS
# ------------------------------------------------------------
print("\n========== RAG QUERY 2 ==========")
results = collection.query(
    query_texts=["how do search engines use embeddings"],
    n_results=3,
    include=["documents", "metadatas", "distances"]
)

for doc, meta, distance in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
    print(f"Distance: {distance:.4f} | Source: {meta['source']} | Chunk #{meta['chunk_index']}")
    print(doc)
    print("-" * 60)

# ------------------------------------------------------------
# 14. FILTER RETRIEVAL TO ONE DOCUMENT
# ------------------------------------------------------------
print("\n========== FILTERED QUERY ==========")
results = collection.query(
    query_texts=["what is deep learning"],
    n_results=2,
    where={"source": "ai_overview.txt"},
    include=["documents", "metadatas", "distances"]
)

for doc, meta, distance in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
    print(f"Distance: {distance:.4f} | Source: {meta['source']} | Chunk #{meta['chunk_index']}")
    print(doc)
    print("-" * 60)

# ------------------------------------------------------------
# 15. FINAL SUMMARY
# ------------------------------------------------------------
print("\n================================================")
print("SUMMARY: ChromaDB storage and querying complete!")
print(f"Final collection size: {collection.count()} chunks.")
print("================================================")

Document created: docs/ai_overview.txt

Number of chunks created: 9

--- Chunk 0 (36 chars) ---
Artificial Intelligence: An Overview

--- Chunk 1 (270 chars) ---
Artificial Intelligence, or AI, is the field of computer science focused on building systems
that can perform tasks which normally require human intelligence. These tasks include
understanding language, recognizing images, making decisions, and learning from experience.

--- Chunk 2 (291 chars) ---
Machine Learning is a subset of AI where systems learn patterns from data instead of following
hardcoded rules. Instead of programming every decision by hand, we feed the system many examples,
and it learns the underlying pattern on its own. This approach powers everything from spam filters

--- Chunk 3 (26 chars) ---
to recommendation systems.

--- Chunk 4 (260 chars) ---
Deep Learning takes Machine Learning further by using neural networks with many layers. These
layered networks can automatically discover complex patterns in larg

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Collection created: rag_chunks

Total chunks stored in ChromaDB: 9

========== STORED CHUNKS ==========
ID: ai_overview_chunk_0
Metadata: {'source': 'ai_overview.txt', 'char_count': 36, 'chunk_index': 0}
Text: Artificial Intelligence: An Overview...
------------------------------------------------------------
ID: ai_overview_chunk_1
Metadata: {'source': 'ai_overview.txt', 'char_count': 270, 'chunk_index': 1}
Text: Artificial Intelligence, or AI, is the field of computer science focused on building systems
that ca...
------------------------------------------------------------
ID: ai_overview_chunk_2
Metadata: {'source': 'ai_overview.txt', 'chunk_index': 2, 'char_count': 291}
Text: Machine Learning is a subset of AI where systems learn patterns from data instead of following
hardc...
------------------------------------------------------------
ID: ai_overview_chunk_3
Metadata: {'char_count': 26, 'chunk_index': 3, 'source': 'ai_overview.txt'}
Text: to recommendation systems....
--------